# 02 · The naive model — good fit ≠ good attribution

A raw-spend OLS (no adstock/saturation/season). It can post a respectable R² and still
attribute wildly, dumping carryover into the intercept.

In [ ]:
import json, numpy as np, pandas as pd
df = pd.read_csv('data/national_weekly.csv', parse_dates=['week'])
chans = ['tv','search','social','affiliate','brand']
X = np.column_stack([df[f'{c}_spend'] for c in chans] + [np.arange(len(df)), np.ones(len(df))])
coef, *_ = np.linalg.lstsq(X, df.conversions.values, rcond=None)
yhat = X @ coef; r2 = 1 - ((df.conversions-yhat)**2).sum()/((df.conversions-df.conversions.mean())**2).sum()
print('naive R^2 =', round(r2,3), '| intercept =', round(coef[-1]))

In [ ]:
# The eval scorecard (produced later by evaluate.py) carries the truth for contrast.
sc = json.load(open('docs/data/scorecard.json'))['naive']
for c in sc['channels']:
    print(f"{c['channel']:10s} true={c['true_contrib']:7.1f}  naive={c['naive_contrib']:8.1f}")

Adding the *true* season as a control makes it **worse**, not better — functional form
(adstock + saturation) matters more than piling on controls.